In [ ]:
dspy.ChainOfThoughtWithHint

In [ ]:
from pprint import pprint
print(result_prompt.prompt.model_dump_json())

{"system_prompt":{"role":"Email Classification Agent","objective":"Analyze incoming emails to determine if they should be processed through internal business systems (IN_SCOPE) or handled through specialized external channels (OUT_OF_SCOPE).","model_instructions":[{"point":"Carefully analyze the email's sender, recipients, subject, and body content to determine if it's an internal business communication or an external communication requiring specialized handling."},{"point":"Look for specific indicators such as external domain names, [EXTERNAL] tags, vendor terminology, and content that suggests the email is from a third-party service provider."},{"point":"Assess whether the email requires standard internal processing or specialized handling by specific departments (procurement, HR, IT)."},{"point":"Provide a confidence score between 0 and 1 that reflects your certainty in the classification, with higher scores indicating greater confidence."},{"point":"Provide a clear rationale for yo

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Annotated


class ClassificationPrompt(BaseModel):
    system_prompt: str = Field(description="The system prompt for the classification.")
    user_prompt: str = Field(description="The user prompt for the classification.")


class ClassificationRationaleSignature(dspy.Signature):
    """Find the correct Rationale for the given Classification. The prompt which classified the input is provided as context and the input is the email body and subject."""

    prompt: ClassificationPrompt = dspy.InputField(
        desc="The prompt which classified the input."
    )
    email_body: str = dspy.InputField(desc="The email body.")
    email_subject: str = dspy.InputField(desc="The email subject.")
    classification_label: str = dspy.InputField(desc="The classification label.")
    classification_rationale: str = dspy.OutputField(desc="The rationale for the classification.")


class ClassificationRationale(dspy.Module):
    def __init__(
        self, prompt: ClassificationPrompt, classification_label: str = "OUT_OF_SCOPE"
    ):
        self.context = prompt
        self.classification_label = classification_label
        self.rationale_signature = dspy.ReAct(ClassificationRationaleSignature,tools=[])
        self.hint="The classification rationale is the explanation for the classification decision."

    def forward(self, email_body: str, email_subject: str) -> str:
        return self.rationale_signature(
            prompt=self.context,
            email_body=email_body,
            email_subject=email_subject,
            classification_label=self.classification_label,
        )


current_prompt = ClassificationPrompt(system_prompt=system_prompt, user_prompt=user_prompt)
rationale_finder = ClassificationRationale(prompt=current_prompt, classification_label="OUT_OF_SCOPE")

In [ ]:
from joblib import Parallel, delayed
from tqdm import trange


all_samples = []

for idx in trange(pure_out_of_scope.shape[0]):
    sample = pure_out_of_scope.iloc[idx].to_dict()
    all_samples.append(sample)

In [ ]:
from tqdm import tqdm

def process_sample(sample):
    lm = dspy.LM(
        model="bedrock/us.anthropic.claude-sonnet-4-20250514-v1:0",
        max_tokens=4096,
        temperature=0.0,
        top_p=1.0,
        top_k=250,
    )
    dspy.settings.configure(lm=lm)
    current_prompt = ClassificationPrompt(system_prompt=system_prompt, user_prompt=user_prompt)
    rationale_finder = ClassificationRationale(prompt=current_prompt, classification_label="OUT_OF_SCOPE")
    rationale = rationale_finder(email_body = sample["body"], email_subject = sample["subject"]).classification_rationale
    sample["better_rationale"] = rationale
    return sample


# results = Parallel(n_jobs=3, backend='threading')(delayed(process_sample)(sample) for sample in tqdm(all_samples))

In [ ]:
# import json

# with open('out_of_scope_samples.json', 'w') as f:
#     json.dump(all_samples, f)

#### Prompt Optimization